In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import chess.pgn
import chess
import numpy as np
import random
import os

CONFIGURATION

In [ ]:
PGN_FILE = "Fischer.pgn"               # Your PGN file
PLAYER_NAME = "Fischer, R"             # Exact name from headers
BATCH_SIZE = 512                       # Cranked up for GPU utilization (reduce to 256 if OOM)
EPOCHS = 50                            
LEARNING_RATE = 1e-3
LABEL_SMOOTHING = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def board_to_tensor(board: chess.Board) -> torch.Tensor:
    piece_map = {chess.PAWN:0, chess.KNIGHT:1, chess.BISHOP:2,
                 chess.ROOK:3, chess.QUEEN:4, chess.KING:5}
    tensor = torch.zeros(19, 8, 8, dtype=torch.float32)

    # Pieces (0-11)
    for sq, p in board.piece_map().items():
        row, col = divmod(sq, 8)
        ch = piece_map[p.piece_type] if p.color == chess.WHITE else 6+piece_map[p.piece_type]
        tensor[ch, row, col] = 1.0

    # Side to move (12)
    if board.turn == chess.WHITE:
        tensor[12,:,:] = 1.0

    # En passant (13)
    if board.ep_square is not None:
        r, c = divmod(board.ep_square, 8)
        tensor[13, r, c] = 1.0

    # Castling rights (14-17)
    cr = board.castling_rights
    tensor[14,:,:] = 1.0 if cr & chess.BB_H1 else 0.0
    tensor[15,:,:] = 1.0 if cr & chess.BB_A1 else 0.0
    tensor[16,:,:] = 1.0 if cr & chess.BB_H8 else 0.0
    tensor[17,:,:] = 1.0 if cr & chess.BB_A8 else 0.0

    # Fifty-move counter (18)
    tensor[18,:,:] = board.halfmove_clock / 100.0

    return tensor

In [ ]:
def move_to_policy_index(move: chess.Move):
    fr, fc = divmod(move.from_square, 8)
    tr, tc = divmod(move.to_square, 8)
    dr, dc = tr - fr, tc - fc

    # Knight moves
    knight_offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
    if (abs(dr), abs(dc)) in [(1,2), (2,1)]:
        try:
            idx = knight_offsets.index((dr, dc))
            return 56 + idx, tr, tc
        except ValueError:
            return None

    # Queen-like moves
    dirs = [(-1,0), (-1,1), (0,1), (1,1), (1,0), (1,-1), (0,-1), (-1,-1)]
    for dir_idx, (ddr, ddc) in enumerate(dirs):
        if (ddr == 0 and ddc == 0) or (dr == 0 and dc == 0):
            continue
        if ddr != 0 and ddc != 0:  # diagonal
            if abs(dr) != abs(dc): continue
            if dr//abs(dr) != ddr or dc//abs(dc) != ddc: continue
            dist = abs(dr)
        elif ddr == 0:  # horizontal
            if dr != 0: continue
            if dc//abs(dc) != ddc: continue
            dist = abs(dc)
        else:  # vertical
            if dc != 0: continue
            if dr//abs(dr) != ddr: continue
            dist = abs(dr)
        if 1 <= dist <= 7:
            plane = dir_idx * 7 + (dist - 1)
            return plane, tr, tc

    # Underpromotions
    if move.promotion and move.promotion != chess.QUEEN:
        if tr == 7:        # White
            forward_dr = -1
            left_dc = -1
            right_dc = 1
        elif tr == 0:      # Black
            forward_dr = 1
            left_dc = 1
            right_dc = -1
        else:
            return None

        if dr == forward_dr and dc == 0:
            dir_idx = 1   # forward
        elif dr == forward_dr and dc == left_dc:
            dir_idx = 0   # left
        elif dr == forward_dr and dc == right_dc:
            dir_idx = 2   # right
        else:
            return None

        if move.promotion == chess.KNIGHT:
            piece_offset = 0
        elif move.promotion == chess.BISHOP:
            piece_offset = 3
        elif move.promotion == chess.ROOK:
            piece_offset = 6
        else:
            return None

        plane = 64 + piece_offset + dir_idx
        return plane, tr, tc

    return None

In [ ]:
class ChessDataset(Dataset):
    def __init__(self, positions, policy_targets):
        self.positions = positions            
        self.policy_targets = policy_targets  

    def __len__(self):
        return len(self.positions)

    def __getitem__(self, idx):
        return self.positions[idx], torch.tensor(self.policy_targets[idx], dtype=torch.long)

def parse_pgn_policy(file_path, player_name):
    positions, targets = [], []
    with open(file_path, encoding='utf-8', errors='ignore') as f:
        while True:
            game = chess.pgn.read_game(f)
            if game is None:
                break
            white = game.headers.get("White", "")
            black = game.headers.get("Black", "")
            target_color = None
            if player_name in white:
                target_color = chess.WHITE
            elif player_name in black:
                target_color = chess.BLACK
            else:
                continue

            board = game.board()
            for move in game.mainline_moves():
                if board.turn == target_color:
                    tensor = board_to_tensor(board)
                    idx = move_to_policy_index(move)
                    if idx is not None:
                        plane, r, c = idx
                        flat = plane * 64 + r * 8 + c
                        positions.append(tensor)
                        targets.append(flat)
                board.push(move)
    return positions, targets

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = torch.relu(x + residual)
        return x

class ChessNet(nn.Module):
    def __init__(self, input_channels=19, num_blocks=6):
        super().__init__()
        self.conv_input = nn.Sequential(
            nn.Conv2d(input_channels, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.res_blocks = nn.Sequential(*[ResidualBlock(128) for _ in range(num_blocks)])
        
        # FIX: AlphaZero-style Convolutional Policy Head
        # Drops model size by 9+ million parameters, preserving 2D spatial locality
        self.policy_head = nn.Sequential(
            nn.Conv2d(128, 73, kernel_size=1, bias=False), 
            nn.BatchNorm2d(73),
            nn.ReLU(),
            nn.Flatten() # 73 channels * 8 * 8 squares = 4672 outputs
        )

    def forward(self, x):
        x = self.conv_input(x)
        x = self.res_blocks(x)
        policy = self.policy_head(x)
        return policy

In [ ]:
def train_model(model, train_loader, val_loader, epochs, lr, device):
    model = model.to(device)
    # OPTIMIZATION: Highly optimized native CrossEntropy with native label smoothing
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    # OPTIMIZATION: AdamW separates weight decay properly for deeper residual setups
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    # OPTIMIZATION: AMP (Automatic Mixed Precision) setup
    use_amp = (device.type == 'cuda')
    scaler = torch.amp.GradScaler('cuda') if use_amp else None

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            
            if use_amp:
                with torch.amp.autocast('cuda'):
                    logits = model(x)
                    loss = criterion(logits, y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(x)
                loss = criterion(logits, y)
                loss.backward()
                optimizer.step()
            
            total_loss += loss.item() * x.size(0)
            _, pred = torch.max(logits, 1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            
        train_acc = correct / total
        scheduler.step()

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                
                if use_amp:
                    with torch.amp.autocast('cuda'):
                        logits = model(x)
                else:
                    logits = model(x)
                    
                _, pred = torch.max(logits, 1)
                val_correct += (pred == y).sum().item()
                val_total += y.size(0)
        val_acc = val_correct / val_total
        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {total_loss/total:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
def pick_move(model, board, temperature=0.5, device='cpu'):
    """
    Highly accelerated move picker. Obtains neural net outputs once
    and applies a vector lookup over legal options to bypass Python latency loops.
    """
    model.eval()
    with torch.no_grad():
        tensor = board_to_tensor(board).unsqueeze(0).to(device)
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    
    legal_moves = list(board.legal_moves)
    if not legal_moves:
        return random.choice(list(board.legal_moves)) if len(board.legal_moves) > 0 else None
        
    move_probs = []
    for move in legal_moves:
        idx = move_to_policy_index(move)
        if idx is not None:
            plane, r, c = idx
            flat_idx = plane * 64 + r * 8 + c
            move_probs.append(probs[flat_idx])
        else:
            move_probs.append(0.0)
            
    move_probs = np.array(move_probs)
    total = move_probs.sum()
    if total > 0:
        move_probs /= total
    else:
        return random.choice(legal_moves)
        
    if temperature > 0:
        move_probs = np.exp(np.log(move_probs + 1e-9) / temperature)
        move_probs /= move_probs.sum()
        return np.random.choice(legal_moves, p=move_probs)
    else:
        return legal_moves[np.argmax(move_probs)]




In [ ]:
def play_vs_human(model, device='cpu', model_plays_white=True):
    board = chess.Board()
    while not board.is_game_over(claim_draw=True):
        print("\n", board)
        if (board.turn == chess.WHITE) == model_plays_white:
            move = pick_move(model, board, temperature=0.1, device=device)
            print(f"Model plays: {move}")
            board.push(move)
        else:
            while True:
                uci = input("Your move (UCI): ")
                try:
                    board.push_uci(uci)
                    break
                except ValueError:
                    print("Illegal move.")
    print("\nGame over:", board.result())

In [ ]:

def self_play(model, device='cpu', temperature=0.5):
    board = chess.Board()
    while not board.is_game_over(claim_draw=True):
        move = pick_move(model, board, temperature, device)
        board.push(move)
        print(board)
        print("---")
    print("Result:", board.result())
    return board

In [ ]:
def save_model(model, path="chess_resnet.pth"):
    # Strip torch.compile dynamic wrappers out if saved
    state_dict = model._orig_mod.state_dict() if hasattr(model, '_orig_mod') else model.state_dict()
    torch.save(state_dict, path)

def load_model(model, path="chess_resnet.pth", device='cpu'):
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    return model


In [ ]:
if __name__ == "__main__":
    print("Parsing PGN and creating policy targets...")
    positions, targets = parse_pgn_policy(PGN_FILE, PLAYER_NAME)
    print(f"Training examples: {len(positions)}")
    if len(positions) == 0:
        print("No examples – check PGN file and player name.")
        exit()

    # Dataset split
    full_dataset = ChessDataset(positions, targets)
    n_train = int(0.8 * len(full_dataset))
    n_val = len(full_dataset) - n_train
    train_set, val_set = torch.utils.data.random_split(full_dataset, [n_train, n_val])
    
    # SAFELY CHECK CUDA FUNCTIONALITY
    # This prevents the script from trying to use a broken/old CUDA configuration
    cuda_is_working = torch.cuda.is_available() and (torch.cuda.get_device_capability(0)[0] >= 3)
    
    if not cuda_is_working:
        print("⚠️ CUDA driver mismatch detected or unavailable. Falling back safely to CPU mode...")
        DEVICE = torch.device("cpu")
        CURRENT_BATCH_SIZE = 64  # Lower batch size so the CPU cache isn't overwhelmed
        NUM_WORKERS = 0          # Set to 0 to avoid Windows multiprocessing bottlenecks on CPU
        PIN_MEMORY = False       # No need to pin memory if we aren't transferring data to a GPU
    else:
        DEVICE = torch.device("cuda")
        CURRENT_BATCH_SIZE = BATCH_SIZE  # Keep 512 for GPU parallel processing
        NUM_WORKERS = 2
        PIN_MEMORY = True

    # Updated DataLoaders with safe system settings
    train_loader = DataLoader(train_set, batch_size=CURRENT_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader = DataLoader(val_set, batch_size=CURRENT_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    # Instantiate Model
    model = ChessNet(input_channels=19, num_blocks=6)
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Only attempt to compile if running on a working CUDA device
    if DEVICE.type == 'cuda':
        print("Compiling model graph via torch.compile for extra speed...")
        model = torch.compile(model)

    # Train
    train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE, device=DEVICE)

    # Save
    save_model(model)

    # Interactive game
    print("\nPlay a game! (Model as White)")
    play_vs_human(model, device=DEVICE, model_plays_white=True)

Epoch 14/50 | Loss: 1.5680 | Train Acc: 0.9164 | Val Acc: 0.2961
